# Day 15 — Batch Pipeline Project

**What you will learn:**
- End-to-end batch ETL following the **Medallion Architecture** (Bronze → Silver → Gold)
- Reading raw data, applying quality rules, writing partitioned Parquet
- Schema enforcement, null handling, deduplication in a real pipeline
- Idempotent writes with `overwrite` mode + partition pruning

**Exam relevance:** Architecture (20%) + DataFrame API (30%) — the medallion pattern, partition strategies, and writing idempotent pipelines are tested.

In [ ]:
from pyspark.sql.functions import (
    col, to_date, to_timestamp, year, month, dayofmonth,
    trim, upper, lower, regexp_replace, when, coalesce, lit,
    current_timestamp, count, countDistinct, sum, avg, round
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
    DateType, TimestampType
)
import tempfile, os

# Use a temp directory as our "data lake" root
LAKE_ROOT = tempfile.mkdtemp(prefix="spark_lake_")
BRONZE = os.path.join(LAKE_ROOT, "bronze")
SILVER = os.path.join(LAKE_ROOT, "silver")
GOLD   = os.path.join(LAKE_ROOT, "gold")

print(f"Lake root: {LAKE_ROOT}")

## 1. Medallion Architecture Overview

```
Raw source
    │
    ▼
BRONZE — raw data, minimal processing, schema-on-read
         Append-only. Every source record lands here.
    │
    ▼
SILVER — cleaned, validated, deduplicated
         Schema enforced. Null checks applied.
    │
    ▼
GOLD   — aggregated, business-ready
         Optimised for BI queries. Read by dashboards.
```

> **Exam tip:** Bronze = raw copy. Silver = trusted data. Gold = aggregated analytics. Each layer is re-derivable from the one above.

## 2. Ingest Raw Data → Bronze

In production this would read from a source system (S3, JDBC, Kafka). Here we simulate messy raw data.

In [ ]:
# Simulated raw sales data — messy strings, nulls, duplicates
raw_schema = StructType([
    StructField("order_id",   StringType(), True),
    StructField("customer",   StringType(), True),
    StructField("product",    StringType(), True),
    StructField("category",   StringType(), True),
    StructField("amount",     StringType(), True),  # raw = string
    StructField("order_date", StringType(), True),  # raw = string
    StructField("region",     StringType(), True),
])

raw_data = [
    ("ORD-001", "  Alice Smith ",  "Laptop",  "Electronics", "1299.99", "2024-01-15", "EU"),
    ("ORD-002", "bob jones",       "Mouse",   "Electronics", "29.99",   "2024-01-15", "US"),
    ("ORD-003", "CAROL O'BRIEN",   "Desk",    "Furniture",   "450.00",  "2024-01-16", "EU"),
    ("ORD-004", "Dave Lee",        "Monitor", "Electronics", "899.50",  "2024-01-16", "APAC"),
    ("ORD-005", "Eve Chen",        "Chair",   "Furniture",   "350.00",  "2024-01-17", "APAC"),
    ("ORD-006", "Alice Smith",     "Keyboard","Electronics", "79.99",   "2024-01-17", "EU"),
    ("ORD-002", "bob jones",       "Mouse",   "Electronics", "29.99",   "2024-01-15", "US"),   # duplicate
    ("ORD-007", None,              "Webcam",  "Electronics", None,      "2024-01-18", "US"),   # nulls
    ("ORD-008", "Frank Miller",    "Laptop",  "Electronics", "INVALID", "2024-01-18", "EU"),   # bad amount
    ("ORD-009", "Grace Park",      "Lamp",    "Furniture",   "89.99",   "2024-01-19", "APAC"),
    ("ORD-010", "Hank Brown",      "Laptop",  "Electronics", "1199.00", "2024-01-20", "US"),
]

df_raw = spark.createDataFrame(raw_data, schema=raw_schema)
print(f"Raw rows: {df_raw.count()}")
df_raw.show(truncate=False)

In [ ]:
# Bronze: land raw data as-is, add ingestion timestamp, partition by date
df_bronze = df_raw.withColumn("_ingested_at", current_timestamp()) \
                  .withColumn("_date_part",   to_date(col("order_date"), "yyyy-MM-dd"))

df_bronze.write \
    .mode("overwrite") \
    .partitionBy("_date_part") \
    .parquet(BRONZE)

print("Bronze written.")
spark.read.parquet(BRONZE).printSchema()

## 3. Bronze → Silver: Clean & Validate

In [ ]:
df_b = spark.read.parquet(BRONZE)
print(f"Bronze rows: {df_b.count()}")

# Step 1: Normalize strings
df_clean = df_b \
    .withColumn("customer", trim(col("customer"))) \
    .withColumn("customer", regexp_replace(col("customer"), r"[^a-zA-Z ]", "")) \
    .withColumn("customer", when(col("customer") == "", None).otherwise(col("customer"))) \
    .withColumn("category", upper(trim(col("category")))) \
    .withColumn("region",   upper(trim(col("region"))))

# Step 2: Cast types — coerce bad amount to null
df_typed = df_clean \
    .withColumn("amount",     col("amount").cast(DoubleType())) \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

# Step 3: Apply quality rules
df_valid = df_typed.filter(
    col("order_id").isNotNull() &
    col("customer").isNotNull() &
    col("amount").isNotNull() &
    (col("amount") > 0)
)

# Step 4: Deduplicate on order_id (keep first occurrence)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w_dedup = Window.partitionBy("order_id").orderBy("_ingested_at")
df_dedup = df_valid.withColumn("_rn", row_number().over(w_dedup)) \
                   .filter(col("_rn") == 1) \
                   .drop("_rn", "_ingested_at")

print(f"Silver rows (after quality + dedup): {df_dedup.count()}")
df_dedup.show(truncate=False)

In [ ]:
# Write Silver — partition by region for efficient BI queries
df_dedup.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(SILVER)

print("Silver written.")
spark.read.parquet(SILVER).printSchema()

## 4. Silver → Gold: Aggregate for BI

In [ ]:
df_s = spark.read.parquet(SILVER)

# Gold table 1: daily revenue by region and category
gold_daily = df_s.groupBy("order_date", "region", "category") \
    .agg(
        count("order_id").alias("order_count"),
        round(sum("amount"), 2).alias("total_revenue"),
        round(avg("amount"), 2).alias("avg_order_value"),
    ) \
    .orderBy("order_date", "region", "category")

gold_daily.show()

gold_daily.write.mode("overwrite").partitionBy("region").parquet(os.path.join(GOLD, "daily_revenue"))
print("Gold daily_revenue written.")

In [ ]:
# Gold table 2: customer lifetime value
gold_clv = df_s.groupBy("customer") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("amount"), 2).alias("lifetime_value"),
        countDistinct("category").alias("categories_bought"),
    ) \
    .orderBy(col("lifetime_value").desc())

gold_clv.show()

gold_clv.write.mode("overwrite").parquet(os.path.join(GOLD, "customer_ltv"))
print("Gold customer_ltv written.")

## 5. Verifying the Lake

In [ ]:
print("=== BRONZE ===")
b = spark.read.parquet(BRONZE)
print(f"Rows: {b.count()}, Partitions: {b.rdd.getNumPartitions()}")

print("\n=== SILVER ===")
s = spark.read.parquet(SILVER)
print(f"Rows: {s.count()}, Partitions: {s.rdd.getNumPartitions()}")

print("\n=== GOLD daily_revenue ===")
g = spark.read.parquet(os.path.join(GOLD, "daily_revenue"))
print(f"Rows: {g.count()}")

print("\n=== GOLD customer_ltv ===")
g2 = spark.read.parquet(os.path.join(GOLD, "customer_ltv"))
g2.show()

---

## Day 15 Checklist

- [ ] Understand the 3-layer medallion architecture and what each layer guarantees
- [ ] Landed raw data to Bronze with ingestion timestamp and date partition
- [ ] Applied string cleaning, type casting, null filtering in Silver
- [ ] Deduplicated using `row_number()` window function
- [ ] Wrote Silver partitioned by region for efficient reads
- [ ] Built two Gold tables: daily revenue and customer LTV
- [ ] Know that `mode("overwrite")` + `partitionBy` makes writes idempotent

**Next:** Day 16 — Spark UI & Troubleshooting